In [1]:
!pip install pyspark

In [2]:
# Initiate PySpark session
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_PySpark_MapReduce") \
    .getOrCreate()

# Check to see if it worked
spark.getActiveSession()

In [3]:
# Download wordcount.txt untuk studi kasus
!wget https://raw.githubusercontent.com/nivdul/spark-in-practice-scala/refs/heads/master/data/wordcount.txt

--2026-06-28 08:13:02--  https://raw.githubusercontent.com/nivdul/spark-in-practice-scala/refs/heads/master/data/wordcount.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4921 (4.8K) [text/plain]
Saving to: ‘wordcount.txt’

wordcount.txt       100%[===================>]   4.81K  --.-KB/s    in 0s      

2026-06-28 08:13:02 (65.6 MB/s) - ‘wordcount.txt’ saved [4921/4921]



In [4]:
# CONTOH #1: MENGHITUNG FREKUENSI KATA DENGAN MAPREDUCE

# Load text file ke format RDD (https://spark.apache.org/docs/latest/rdd-programming-guide.html#resilient-distributed-datasets-rdds)
inputRDD = spark.sparkContext.textFile("wordcount.txt")

# Cek jumlah partisi/split
print('The number of partitions: ',inputRDD.getNumPartitions(), '\nThe total number of elements: ', inputRDD.count())

# Print isi 10 pertama
print(inputRDD.take(10))

The number of partitions:  2 
The total number of elements:  44
['word count from Wikipedia the free encyclopedia', 'the word count is the number of words in a document or passage of text Word counting may be needed when a text', 'is required to stay within certain numbers of words This may particularly be the case in academia legal', 'proceedings journalism and advertising Word count is commonly used by translators to determine the price for', 'the translation job Word counts may also be used to calculate measures of readability and to measure typing', 'and reading speeds usually in words per minute When converting character counts to words a measure of five or', 'six characters to a word is generally used Contents Details and variations of definition Software In fiction', 'In non fiction See also References Sources External links Details and variations of definition', 'This section does not cite any references or sources Please help improve this section by adding citations to', 'reli

In [5]:
# Flat Mapper mengembalikan flat data dengan separator spasi
words = inputRDD.flatMap(lambda x: x.split(' '))
print(words.take(10))

# MAPPER untuk menyimpan nilai 1 untuk setiap kemunculan
wordsOne = words.map(lambda x: (x, 1))
print(wordsOne.take(10))

['word', 'count', 'from', 'Wikipedia', 'the', 'free', 'encyclopedia', 'the', 'word', 'count']
[('word', 1), ('count', 1), ('from', 1), ('Wikipedia', 1), ('the', 1), ('free', 1), ('encyclopedia', 1), ('the', 1), ('word', 1), ('count', 1)]


In [6]:
from operator import add

# REDUCE untuk menjumlahkan (ADD) kemunculan setiap kata yang sudah disimpan di MAPPER
wordCounts = wordsOne.reduceByKey(add)
#print(wordCounts.take(10))

In [7]:
# Pengumpulan hasil (merging hasil-hasil dari reducer)
output = wordCounts.collect()
# Print top-10
output[:10]

[('word', 24),
 ('from', 2),
 ('encyclopedia', 1),
 ('number', 3),
 ('of', 25),
 ('words', 21),
 ('text', 8),
 ('Word', 3),
 ('counting', 6),
 ('needed', 1)]

In [8]:
# Tampilkan top 10 terurut
sorted(output, key=lambda x: x[1], reverse = True)[0:10]

[('the', 38),
 ('a', 28),
 ('of', 25),
 ('word', 24),
 ('and', 23),
 ('words', 21),
 ('is', 19),
 ('to', 18),
 ('count', 11),
 ('in', 11)]

In [9]:
# CONTOH #2: MENCARI IKAN TERBERAT
import kagglehub, os

os.environ['KAGGLEHUB_CACHE'] = "/content/kaggle"
# Download fish dataset
path = kagglehub.dataset_download("vipullrathod/fish-market")

print("Path to dataset files:", path)

100%|██████████| 2.38k/2.38k [00:00<00:00, 4.45MB/s]

Extracting files...
Path to dataset files: /content/kaggle/datasets/vipullrathod/fish-market/versions/1


In [10]:
import pyspark.sql.functions as F

# 1. Load the fish.csv ke dataframe
df = spark.read.csv(path + "/Fish.csv", header=True, inferSchema=True)

# 2. Cari ikan terberat dengan groupby (tanpa MapReduce)
max_weight_per_species = df.groupBy("Species").agg(F.max("Weight").alias("MaxWeight"))

# 3. Tampilkan hasil
max_weight_per_species.show()

+---------+---------+
|  Species|MaxWeight|
+---------+---------+
|    Roach|    390.0|
|    Smelt|     19.9|
|   Parkki|    300.0|
|Whitefish|   1000.0|
|     Pike|   1650.0|
|    Bream|   1000.0|
|    Perch|   1100.0|
+---------+---------+



In [11]:
# TUGAS
# Cari 3 species ikan terberat menggunakan map-reduce seperti di contoh #1 wordcount.
# Pastikan daftar spesies sama dengan yang dihasilkan contoh dengan dataframe (tanpa mapreduce).
# Tips: load dataframe (df) ke spark RDD terlebih dahulu.
# Buat dan upload laporan dalam bentuk 1-2 halaman PDF yang mencakup strategi kamu di fungsi map dan reduce, kode dan penjelasannya.
# Presentasikan dalam kelompok

## Tugas: 3 Species Ikan Terberat dengan MapReduce

Strategi mengikuti contoh WordCount:
1. **Map**: ubah setiap baris menjadi pasangan `(Species, Weight)`.
2. **Reduce**: gabungkan data berdasarkan `Species` dan ambil `Weight` maksimum.
3. **Output**: urutkan hasil berdasarkan berat terbesar, lalu ambil 3 species teratas.
4. **Validasi**: hasil MapReduce dibandingkan dengan hasil DataFrame biasa.


In [12]:
# Cell 1 - Baseline dengan DataFrame tanpa MapReduce
# Tujuan: menghasilkan pembanding yang benar dari DataFrame

import pyspark.sql.functions as F

top3_df = (
    df.select("Species", "Weight")
      .dropna()
      .groupBy("Species")
      .agg(F.max("Weight").alias("MaxWeight"))
      .orderBy(F.desc("MaxWeight"), F.asc("Species"))
      .limit(3)
)

top3_df.show()

top3_df_list = [(row["Species"], row["MaxWeight"]) for row in top3_df.collect()]
top3_df_list


+-------+---------+
|Species|MaxWeight|
+-------+---------+
|   Pike|   1650.0|
|  Perch|   1100.0|
|  Bream|   1000.0|
+-------+---------+



[('Pike', 1650.0), ('Perch', 1100.0), ('Bream', 1000.0)]

In [13]:
# Cell 2 - Load DataFrame ke Spark RDD
# Tips dari soal: dataframe (df) dimasukkan ke RDD terlebih dahulu

fish_rdd = df.select("Species", "Weight").dropna().rdd

print("Jumlah partisi RDD:", fish_rdd.getNumPartitions())
print("Jumlah data:", fish_rdd.count())
fish_rdd.take(5)


Jumlah partisi RDD: 1
Jumlah data: 159


[Row(Species='Bream', Weight=242.0),
 Row(Species='Bream', Weight=290.0),
 Row(Species='Bream', Weight=340.0),
 Row(Species='Bream', Weight=363.0),
 Row(Species='Bream', Weight=430.0)]

In [14]:
# Cell 3 - MAP
# Mapper menghasilkan pasangan key-value: (Species, Weight)

mapped_rdd = fish_rdd.map(lambda row: (row["Species"], float(row["Weight"])))

mapped_rdd.take(10)


[('Bream', 242.0),
 ('Bream', 290.0),
 ('Bream', 340.0),
 ('Bream', 363.0),
 ('Bream', 430.0),
 ('Bream', 450.0),
 ('Bream', 500.0),
 ('Bream', 390.0),
 ('Bream', 450.0),
 ('Bream', 500.0)]

In [15]:
# Cell 4 - REDUCE
# Reducer menggabungkan data per Species dan mengambil Weight maksimum

max_weight_rdd = mapped_rdd.reduceByKey(lambda a, b: a if a >= b else b)

max_weight_rdd.collect()


[('Bream', 1000.0),
 ('Roach', 390.0),
 ('Whitefish', 1000.0),
 ('Parkki', 300.0),
 ('Perch', 1100.0),
 ('Pike', 1650.0),
 ('Smelt', 19.9)]

In [16]:
# Cell 5 - Ambil 3 species ikan terberat
# Urutkan berdasarkan Weight terbesar, jika sama urutkan berdasarkan nama Species

top3_mapreduce = max_weight_rdd.sortBy(lambda x: (-x[1], x[0])).take(3)

top3_mapreduce


[('Pike', 1650.0), ('Perch', 1100.0), ('Bream', 1000.0)]

In [17]:
# Cell 6 - Validasi hasil MapReduce dengan hasil DataFrame
# Daftar species harus sama dengan hasil DataFrame tanpa MapReduce

df_species = [species for species, weight in top3_df_list]
mr_species = [species for species, weight in top3_mapreduce]

print("Top 3 DataFrame :", top3_df_list)
print("Top 3 MapReduce :", top3_mapreduce)

if df_species == mr_species:
    print("VALID: daftar species sama dengan hasil DataFrame.")
else:
    print("TIDAK SAMA: cek kembali proses map/reduce.")


Top 3 DataFrame : [('Pike', 1650.0), ('Perch', 1100.0), ('Bream', 1000.0)]
Top 3 MapReduce : [('Pike', 1650.0), ('Perch', 1100.0), ('Bream', 1000.0)]
VALID: daftar species sama dengan hasil DataFrame.


### Penjelasan Singkat

Pada tahap **Map**, setiap baris data ikan diubah menjadi pasangan key-value `(Species, Weight)`.  
Pada tahap **Reduce**, data dengan key species yang sama digabungkan menggunakan fungsi maksimum sehingga diperoleh berat terbesar untuk setiap species.  
Setelah itu hasil MapReduce diurutkan dari berat terbesar dan diambil 3 species teratas.  
Cell validasi digunakan untuk memastikan daftar species dari MapReduce sama dengan hasil pengolahan DataFrame tanpa MapReduce.
